# DS-MNIST - Versión mejorada
Se añadió: más neuronas, capas adicionales, BatchNorm, Dropout, batching, device (GPU si está disponible), scheduler y evaluación por época.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Cargar los datos de MNIST desde OpenML.
# fetch_openml devuelve un objeto con los datos y las etiquetas.
mnist = fetch_openml('mnist_784', version=1, as_frame=False)

# Se convierten los datos a float32 para trabajar mejor con PyTorch.
# y se convierten las etiquetas a enteros para clasificación.
X, y = mnist.data.astype(np.float32), mnist.target.astype(np.int64)

# Normalizar los pixeles a un rango de 0 a 1.
X /= 255.0

# Separar el conjunto en entrenamiento y prueba.
# test_size=0.2 significa que 20% se reserva para prueba.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
 )

# Elegir el dispositivo: GPU si existe, si no CPU.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Convertir los arreglos NumPy a tensores de PyTorch.
X_train_t = torch.from_numpy(X_train)
y_train_t = torch.from_numpy(y_train)
X_test_t = torch.from_numpy(X_test)
y_test_t = torch.from_numpy(y_test)

# Crear datasets para que PyTorch pueda leer pares (entrada, etiqueta).
train_ds = TensorDataset(X_train_t, y_train_t)
test_ds = TensorDataset(X_test_t, y_test_t)

# DataLoader divide el dataset en mini-lotes.
# batch_size controla cuántas muestras se procesan por paso.
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

# Definir una red neuronal feedforward más profunda.
# Tiene dos capas ocultas, BatchNorm y Dropout.
class Net(nn.Module):
    def __init__(self):
        super().__init__()

        # Secuencia de capas del modelo.
        self.net = nn.Sequential(
            nn.Linear(784, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        # x entra como vector de 784 valores y sale como logits de 10 clases.
        return self.net(x)

# Crear el modelo y enviarlo al mismo dispositivo que los datos.
model = Net().to(device)

# CrossEntropyLoss combina softmax + log-loss para clasificación multiclase.
criterion = nn.CrossEntropyLoss()

# Adam ajusta los pesos con una tasa de aprendizaje adaptativa.
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ReduceLROnPlateau baja el learning rate cuando la pérdida deja de mejorar.
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=2, factor=0.5, verbose=True
 )

# Número de épocas de entrenamiento.
epochs = 15

# Listas para guardar historia y luego graficarla.
train_losses = []
test_losses = []
train_accs = []
test_accs = []

# Bucle principal de entrenamiento.
for epoch in range(1, epochs + 1):
    # Modo entrenamiento: activa Dropout y BatchNorm en modo train.
    model.train()

    # Variables para acumular métricas de entrenamiento.
    train_loss = 0.0
    correct = 0
    total = 0

    # Recorre el dataset por mini-lotes.
    for xb, yb in train_loader:
        # Mover lote y etiquetas al mismo dispositivo del modelo.
        xb = xb.to(device)
        yb = yb.to(device)

        # Limpiar gradientes previos antes de la nueva actualización.
        optimizer.zero_grad()

        # Forward pass: el modelo produce logits para cada clase.
        out = model(xb)

        # Calcular la pérdida entre predicción y etiqueta real.
        loss = criterion(out, yb)

        # Backpropagation: calcular gradientes de todos los parámetros.
        loss.backward()

        # Actualizar pesos con Adam.
        optimizer.step()

        # Guardar pérdida acumulada ponderada por el tamaño del lote.
        train_loss += loss.item() * xb.size(0)

        # Obtener la clase predicha con mayor score.
        _, pred = out.max(1)

        # Contar cuántos aciertos hubo en el lote.
        correct += pred.eq(yb).sum().item()
        total += xb.size(0)

    # Promedio de pérdida y accuracy del entrenamiento.
    train_loss /= total
    train_acc = correct / total

    # Guardar historial de entrenamiento.
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # Modo evaluación: desactiva Dropout y usa BatchNorm en modo inferencia.
    model.eval()

    # Variables para acumular métricas de prueba.
    test_loss = 0.0
    correct = 0
    total = 0

    # No calcular gradientes en validación ahorra memoria y tiempo.
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            # Forward pass sobre el conjunto de prueba.
            out = model(xb)

            # Calcular pérdida de prueba.
            loss = criterion(out, yb)
            test_loss += loss.item() * xb.size(0)

            # Clase con mayor probabilidad implícita.
            _, pred = out.max(1)
            correct += pred.eq(yb).sum().item()
            total += xb.size(0)

    # Promedio de pérdida y accuracy de prueba.
    test_loss /= total
    test_acc = correct / total

    # Guardar historial de prueba.
    test_losses.append(test_loss)
    test_accs.append(test_acc)

    # Ajustar learning rate si la pérdida de validación deja de mejorar.
    scheduler.step(test_loss)

    # Mostrar progreso de la época.
    print(
        f'Epoch {epoch}/{epochs} - train_loss: {train_loss:.4f} '
        f'train_acc: {train_acc:.4f} test_loss: {test_loss:.4f} '
        f'test_acc: {test_acc:.4f}'
    )

# Graficar pérdida y accuracy para ver cómo aprende el modelo.
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Test Loss')
plt.title('Loss por época')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Acc')
plt.plot(test_accs, label='Test Acc')
plt.title('Accuracy por época')
plt.xlabel('Época')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

# Mostrar una predicción de ejemplo.
sample_idx = 0
model.eval()
with torch.no_grad():
    # Tomar una sola imagen, agregar dimensión batch y moverla al dispositivo.
    sample = torch.from_numpy(X_test[sample_idx].astype(np.float32)).unsqueeze(0).to(device)
    pred = model(sample).argmax(dim=1).item()

plt.imshow(X_test[sample_idx].reshape(28, 28), cmap='gray')
plt.title(f'Pred: {pred}, True: {y_test[sample_idx]}')
plt.show()